# Phase 2 Verification — Kiểm tra tích hợp Neo4j + Qdrant

Notebook này verify tất cả DoD items của TASK-09. Chạy từng cell theo thứ tự.

**Yêu cầu:** Neo4j và Qdrant đang chạy, file `.env` đã có credentials.

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

load_dotenv()

driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USER'), os.getenv('NEO4J_PASSWORD')),
)
qdrant = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
print('Kết nối thành công.')

Kết nối thành công.


## 1. Neo4j — Node Counts (DoD: 6 loại node đều có count > 0)

In [2]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n) RETURN labels(n)[0] as type, count(n) as count ORDER BY count DESC'
    ).data()

print(f"{'Node type':<15} {'Count':>8}")
print('-' * 25)
for r in rows:
    print(f"{r['type']:<15} {r['count']:>8}")

types_found = {r['type'] for r in rows}
required = {'Theme', 'Norm', 'Component', 'CTV', 'TextUnit', 'Jurisdiction'}
print(f"\n6 loại node đủ: {'✅' if required <= types_found else '❌ thiếu: ' + str(required - types_found)}")

Node type          Count
-------------------------
Component           3014
CTV                 3014
TextUnit            3014
Norm                  17
Jurisdiction           3
Theme                  1

6 loại node đủ: ✅


## 2. Neo4j — [:IMPLEMENTS] chain (DoD: tier 2 → tier 1)

In [3]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n:Norm {tier:2})-[:IMPLEMENTS]->(p:Norm {tier:1}) '
        'RETURN n.id AS from_id, n.tier AS from_tier, p.id AS to_id, p.tier AS to_tier LIMIT 5'
    ).data()

print('[:IMPLEMENTS] chains (tier 2 → tier 1):')
for r in rows:
    print(f"  {r['from_id']} (tier {r['from_tier']}) → {r['to_id']} (tier {r['to_tier']})")
print(f"\nKết quả: {'✅ hợp lệ' if rows else '❌ không có chain'}")

[:IMPLEMENTS] chains (tier 2 → tier 1):
  nghi-dinh-102-2024-nd-cp (tier 2) → luat-dat-dai-2024 (tier 1)
  nghi-dinh-49-2026-nd-cp (tier 2) → nghi-quyet-254-2025-qh15 (tier 1)
  nghi-dinh-50-2026-nd-cp (tier 2) → nghi-quyet-254-2025-qh15 (tier 1)

Kết quả: ✅ hợp lệ


## 3. Neo4j — [:APPLIES_TO] jurisdiction (DoD: jurisdiction đúng)

In [4]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n:Norm)-[:APPLIES_TO]->(j:Jurisdiction) '
        'RETURN n.id AS norm_id, j.name AS jurisdiction LIMIT 10'
    ).data()

print(f"{'norm_id':<45} {'jurisdiction'}")
print('-' * 60)
for r in rows:
    print(f"{r['norm_id']:<45} {r['jurisdiction']}")

norm_id                                       jurisdiction
------------------------------------------------------------
nghi-quyet-254-2025-qh15                      toan-quoc
nghi-dinh-50-2026-nd-cp                       toan-quoc
nghi-dinh-49-2026-nd-cp                       toan-quoc
nghi-dinh-226-2025-nd-cp                      toan-quoc
nghi-dinh-151-2025-nd-cp                      toan-quoc
nghi-dinh-112-2024-nd-cp                      toan-quoc
nghi-dinh-102-2024-nd-cp                      toan-quoc
nghi-dinh-101-2024-nd-cp                      toan-quoc
luat-dat-dai-2024                             toan-quoc
quyet-dinh-69-2024-qd-ubnd-tp-hcm             tp-hcm


## 4. Qdrant — Vector Counts (DoD: counts khớp với Neo4j)

In [5]:
tu_count = qdrant.count(
    'legal_texts',
    count_filter=Filter(must=[FieldCondition(key='content_type', match=MatchValue(value='text_unit'))]),
).count
sm_count = qdrant.count(
    'legal_texts',
    count_filter=Filter(must=[FieldCondition(key='content_type', match=MatchValue(value='summary'))]),
).count

with driver.session() as s:
    neo4j_tu = s.run('MATCH (t:TextUnit) RETURN count(t) AS c').single()['c']
    neo4j_norm = s.run('MATCH (n:Norm) RETURN count(n) AS c').single()['c']

print(f"text_unit vectors : {tu_count:>5} | Neo4j TextUnit: {neo4j_tu:>5} | {'✅' if tu_count == neo4j_tu else '❌'}")
print(f"summary  vectors  : {sm_count:>5} | Neo4j Norm    : {neo4j_norm:>5} | {'✅' if sm_count == neo4j_norm else '❌'}")

text_unit vectors :  3014 | Neo4j TextUnit:  3014 | ✅
summary  vectors  :    17 | Neo4j Norm    :    17 | ✅


## 5. Vector Search — Stage 1 Summary Routing

Query: `"phí chuyển mục đích sử dụng đất"` | Filter: `content_type="summary"`, `theme="dat-dai"`

In [6]:
import sys
sys.path.insert(0, '..')
from src.ingestion.vectorizer import load_model, encode_text

model = load_model()

q1 = 'phí chuyển mục đích sử dụng đất'
vec1 = encode_text(model, q1)
res1 = qdrant.query_points(
    'legal_texts',
    query=vec1,
    limit=3,
    query_filter=Filter(must=[
        FieldCondition(key='content_type', match=MatchValue(value='summary')),
        FieldCondition(key='theme', match=MatchValue(value='dat-dai')),
    ]),
).points

print(f'Stage 1 query: "{q1}"')
print(f"{'Rank':<6} {'Score':<8} {'norm_id':<40} {'tier':<6} {'jurisdiction'}")
print('-' * 80)
for i, r in enumerate(res1, 1):
    print(f"{i:<6} {r.score:<8.4f} {r.payload['norm_id']:<40} {r.payload['tier']:<6} {r.payload['jurisdiction']}")

all_dat_dai = all(r.payload['theme'] == 'dat-dai' for r in res1)
print(f"\nTop-3 đều dat-dai: {'✅' if all_dat_dai else '❌'}")

INFO TensorFlow version 2.19.0 available.
INFO Loading model BAAI/bge-m3 ...
INFO No device provided, using mps
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO Loading SentenceTransformer model from BAAI/bge-m3.
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_trans

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.j

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"


Stage 1 query: "phí chuyển mục đích sử dụng đất"
Rank   Score    norm_id                                  tier   jurisdiction
--------------------------------------------------------------------------------
1      0.6381   nghi-dinh-50-2026-nd-cp                  2      toan-quoc
2      0.5978   nghi-quyet-02-2023-nq-hdnd-tp-hcm        4      tp-hcm
3      0.5828   nghi-quyet-22-2024-nq-hdnd-dong-nai      4      dong-nai

Top-3 đều dat-dai: ✅


## 6. Vector Search — Stage 2 Text Unit Retrieval

Query: `"đăng ký khai sinh"` | Filter: `content_type="text_unit"`, `jurisdiction="toan-quoc"`

> **Lưu ý:** Hiện tại chỉ có dữ liệu Đất đai ([A]). Kết quả sẽ trả về `dat-dai` cho đến khi [B] nộp file Hộ tịch và chạy lại pipeline ingestion.

In [7]:
q2 = 'đăng ký khai sinh'
vec2 = encode_text(model, q2)
res2 = qdrant.query_points(
    'legal_texts',
    query=vec2,
    limit=3,
    query_filter=Filter(must=[
        FieldCondition(key='content_type', match=MatchValue(value='text_unit')),
        FieldCondition(key='jurisdiction', match=MatchValue(value='toan-quoc')),
    ]),
).points

print(f'Stage 2 query: "{q2}"')
print(f"{'Rank':<6} {'Score':<8} {'norm_id':<40} {'theme'}")
print('-' * 70)
for i, r in enumerate(res2, 1):
    print(f"{i:<6} {r.score:<8.4f} {r.payload['norm_id']:<40} {r.payload['theme']}")

print('\n⏳ Re-verify sau khi [B] nộp dữ liệu Hộ tịch.')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO HTTP Request: POST http://localhost:6333/collections/legal_texts/points/query "HTTP/1.1 200 OK"


Stage 2 query: "đăng ký khai sinh"
Rank   Score    norm_id                                  theme
----------------------------------------------------------------------
1      0.8229   nghi-dinh-49-2026-nd-cp                  dat-dai
2      0.7982   luat-dat-dai-2024                        dat-dai
3      0.7954   luat-dat-dai-2024                        dat-dai

⏳ Re-verify sau khi [B] nộp dữ liệu Hộ tịch.


## 7. Idempotency Check

In [8]:
with driver.session() as s:
    node_count = s.run('MATCH (n) RETURN count(n) AS c').single()['c']

total_vectors = qdrant.get_collection('legal_texts').points_count

print(f'Neo4j total nodes : {node_count}')
print(f'Qdrant total vectors: {total_vectors}')
print('\n(Chạy lại run_ingestion() + run_vectorization() rồi so sánh — số không được tăng.)')

INFO HTTP Request: GET http://localhost:6333/collections/legal_texts "HTTP/1.1 200 OK"


Neo4j total nodes : 9063
Qdrant total vectors: 3031

(Chạy lại run_ingestion() + run_vectorization() rồi so sánh — số không được tăng.)


In [9]:
driver.close()
print('Done. Xem phase2_report.md để biết kết quả đầy đủ.')

Done. Xem phase2_report.md để biết kết quả đầy đủ.
